In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import joblib
import os


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import precision_recall_curve, auc, classification_report, confusion_matrix

# Configurações de exibição e reprodutibilidade
pd.set_option('display.max_columns', None)
SEED = 42
np.random.seed(SEED)

print("Ambiente configurado com sucesso.")

In [ ]:
DATA_PATH = "../data/creditcard.csv"

if not os.path.exists(DATA_PATH):
    print(f"ERRO: Dataset não encontrado em {DATA_PATH}. Certifique-se de baixar o arquivo creditcard.csv da ULB.")
else:
    df = pd.read_csv(DATA_PATH)
    print(f"Dataset carregado: {df.shape[0]} linhas e {df.shape[1]} colunas.")
    display(df.head())

In [ ]:
# Proporção das classes
class_counts = df['Class'].value_counts(normalize=True) * 100
print(f"Transações Legítimas: {class_counts[0]:.2f}%")
print(f"Transações Fraudulentas: {class_counts[1]:.2f}%")

# Visualização rápida
fig = px.pie(values=df['Class'].value_counts(), names=['Legítimo', 'Fraude'], 
             title='Distribuição de Classes (Target)', color_discrete_sequence=['#636EFA', '#EF553B'])
fig.show()

In [ ]:
# Aplicando RobustScaler em Time e Amount
rob_scaler = RobustScaler()

df['scaled_amount'] = rob_scaler.fit_transform(df['Amount'].values.reshape(-1,1))
df['scaled_time'] = rob_scaler.fit_transform(df['Time'].values.reshape(-1,1))

# Removendo as colunas originais
df.drop(['Time', 'Amount'], axis=1, inplace=True)

# Reordenando para deixar o target no final e colunas escaladas no início
scaled_amount = df['scaled_amount']
scaled_time = df['scaled_time']
df.drop(['scaled_amount', 'scaled_time'], axis=1, inplace=True)
df.insert(0, 'scaled_amount', scaled_amount)
df.insert(1, 'scaled_time', scaled_time)

print("Escalonamento robusto concluído.")

In [ ]:
X = df.drop('Class', axis=1)
y = df['Class']

# Divisão 80/20 com estratificação
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)

# Geração da Baseline para o Dashboard (Amostra de 5000 registros do treino)
os.makedirs('../artifacts', exist_ok=True)
X_baseline_sample = X_train.sample(n=min(5000, len(X_train)), random_state=SEED)
X_baseline_sample.to_parquet('../artifacts/baseline_features.parquet', index=False)

print(f"Divisão concluída.")
print(f"Treino: {X_train.shape[0]} amostras | Teste: {X_test.shape[0]} amostras")
print("Arquivo artifacts/baseline_features.parquet gerado para monitoramento de drift.")

In [ ]:
# Criando um subconjunto balanceado APENAS para explorar as correlações
# Extraímos todas as fraudes do conjunto de treino e uma amostra igual de não-fraudes
frauds_train = X_train[y_train == 1]
non_frauds_train = X_train[y_train == 0].sample(n=len(frauds_train), random_state=SEED)

balanced_train = pd.concat([frauds_train, non_frauds_train])
y_balanced = pd.concat([y_train[y_train == 1], y_train[y_train == 0].sample(n=len(frauds_train), random_state=SEED)])
df_balanced_plot = pd.concat([balanced_train, y_balanced], axis=1)

# Heatmap de Correlação
plt.figure(figsize=(24,20))
sns.heatmap(df_balanced_plot.corr(), cmap='coolwarm_r', annot_kws={'size':20}, fmt='.2f')
plt.title('Matriz de Correlação (Subamostra Balanceada)', fontsize=14)
plt.show()

# Extraindo as variáveis mais correlacionadas (positiva e negativamente)
correlations = df_balanced_plot.corr()['Class'].sort_values()
print("\nTop Correlações Negativas:\n", correlations.head(4))
print("\nTop Correlações Positivas:\n", correlations.tail(5)[:-1]) # Exclui a correlação de 1.0 com ela mesma

In [ ]:
# Focaremos nas variáveis com alta correlação negativa (exemplo: V14, V12, V10 baseando-se no dataset ULB)
# No mundo real, você extrairia essas colunas dinamicamente do resultado da célula anterior.
features_to_clean = ['V14', 'V12', 'V10']
X_train_clean = X_train.copy()
y_train_clean = y_train.copy()

for feature in features_to_clean:
    # Selecionamos apenas as fraudes para calcular os limites
    frauds = X_train_clean[y_train_clean == 1][feature].values
    q25, q75 = np.percentile(frauds, 25), np.percentile(frauds, 75)
    iqr = q75 - q25
    
    # Limites (usando multiplicador 1.5 para detecção padrão)
    lower_bound = q25 - (1.5 * iqr)
    upper_bound = q75 + (1.5 * iqr)
    
    # Índices dos outliers dentro das fraudes
    outliers_index = X_train_clean[(X_train_clean[feature] < lower_bound) | (X_train_clean[feature] > upper_bound)].index
    outliers_frauds_index = [idx for idx in outliers_index if y_train_clean.loc[idx] == 1]
    
    # Removendo os outliers (apenas no treino)
    X_train_clean = X_train_clean.drop(outliers_frauds_index)
    y_train_clean = y_train_clean.drop(outliers_frauds_index)

print(f"Instâncias originais no treino: {len(X_train)}")
print(f"Instâncias após remoção de outliers (apenas em fraudes): {len(X_train_clean)}")

In [ ]:
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import RobustScaler
from sklearn.compose import ColumnTransformer

# Identificando colunas para o transformer
# Se você já removeu as colunas originais na Célula 4, use 'scaled_amount' e 'scaled_time'.
# Mas o ideal de um pipeline de produção é receber os dados brutos.
features_to_scale = ['Amount', 'Time'] 
other_features = [f'V{i}' for i in range(1, 29)]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', RobustScaler(), features_to_scale),
        ('passthrough', 'passthrough', other_features)
    ])

# Pipeline completo: Escalonamento -> Balanceamento -> Classificação
pipeline = ImbPipeline([
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=SEED)),
    ('classifier', XGBClassifier(tree_method='hist', random_state=SEED, eval_metric='logloss'))
])

# Validação Cruzada
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
print("Iniciando Validação Cruzada com Pipeline Integrado...")
cv_scores = cross_val_score(pipeline, X_train, y_train, cv=kf, scoring='average_precision', n_jobs=-1)

print(f"AUPRC Médio: {np.mean(cv_scores):.4f}")

In [ ]:
# Treinando o pipeline completo com os dados de treino limpos
pipeline.fit(X_train_clean, y_train_clean)

print("Pipeline treinado com sucesso em todo o conjunto de treino.")

In [ ]:
# Predições de probabilidade para a classe positiva (fraude)
y_probs = pipeline.predict_proba(X_test)[:, 1]
y_pred = pipeline.predict(X_test)

# Cálculo da Precisão e Recall para diferentes thresholds
precision, recall, _ = precision_recall_curve(y_test, y_probs)
auprc_score = auc(recall, precision)

print(f"AUPRC Final no Conjunto de Teste: {auprc_score:.4f}")

# Visualização da Curva Precision-Recall
fig_pr = px.area(
    x=recall, y=precision,
    title=f'Curva Precision-Recall (AUPRC={auprc_score:.4f})',
    labels=dict(x='Recall', y='Precision'),
    width=700, height=500
)
fig_pr.add_shape(type='line', line=dict(dash='dash'), x0=0, x1=1, y0=0.5, y1=0.5)
fig_pr.show()

# Matriz de Confusão
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Matriz de Confusão (Threshold Padrão 0.5)')
plt.xlabel('Predito')
plt.ylabel('Real')
plt.show()

In [ ]:
# Célula: Otimização de Threshold por Custo de Negócio
def calculate_business_cost(y_true, y_probs, cost_fn=100, cost_fp=20):
    thresholds = np.linspace(0, 1, 100)
    costs = []
    
    for t in thresholds:
        y_pred_t = (y_probs >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred_t).ravel()
        # Falso Negativo: Fraude que passou (Prejuízo direto)
        # Falso Positivo: Cliente legítimo bloqueado (Custo operacional/atrito)
        total_cost = (fn * cost_fn) + (fp * cost_fp)
        costs.append(total_cost)
        
    best_t = thresholds[np.argmin(costs)]
    return best_t, costs

cost_fn = 500  # Valor médio de uma fraude perdida
cost_fp = 50   # Custo de suporte para desbloqueio/atrito
best_threshold, business_costs = calculate_business_cost(y_test, y_probs, cost_fn, cost_fp)

print(f"Melhor Threshold para o Negócio: {best_threshold:.2f}")

# Visualização do impacto financeiro
fig_cost = px.line(x=np.linspace(0, 1, 100), y=business_costs, 
                  labels={'x': 'Threshold', 'y': 'Custo Estimado (R$)'},
                  title='Otimização Financeira do Ponto de Corte')
fig_cost.add_vline(x=best_threshold, line_dash="dash", line_color="red")
fig_cost.show()

In [ ]:
import shap

# Como usamos um Pipeline, precisamos extrair o modelo treinado dentro dele
explainer = shap.TreeExplainer(pipeline.named_steps['classifier'])
# Calculamos os SHAP values para uma amostra do teste para agilizar na CPU
shap_values = explainer.shap_values(X_test.iloc[:100])

shap.summary_plot(shap_values, X_test.iloc[:100], plot_type="bar")

In [ ]:
# Definindo o caminho de saída
MODEL_PATH = "../artifacts/fraud_model.joblib"

# Exportando o pipeline completo
# O pipeline já contém a lógica de SMOTE (que será ignorada no predict) e o XGBoost
joblib.dump(pipeline, MODEL_PATH)

print(f"Artefato exportado para: {MODEL_PATH}")

In [ ]:
# Supondo que X_train seja o DataFrame utilizado para treinar o modelo
import pandas as pd
import os

# 1. Obter uma amostra representativa (ex: 5000 linhas) para não pesar no deploy
# Usamos stratify=y_train para garantir que a proporção fraude/legítimo se mantenha na baseline
_, X_baseline_sample, _, y_baseline_sample = train_test_split(
    X_train, y_train, test_size=5000, stratify=y_train, random_state=42
)

# 2. Salvar como Parquet (muito menor, mais rápido de ler no Streamlit e preserva tipos de dados, superior ao CSV)
os.makedirs('../artifacts', exist_ok=True)
X_baseline_sample.to_parquet('../artifacts/baseline_features.parquet', index=False)

print("Baseline estatística exportada com sucesso para /artifacts/baseline_features.parquet")